# 결정트리: 질문을 끝없이 세분하면 처음 보는 학생도 더 잘 맞힐까?

조건에 따라 가지를 나누며 분류합니다.

**실행 방법:** Google Colab에서 이 노트북을 업로드한 뒤 위에서 아래로 실행합니다. GPU와 API 키는 필요하지 않습니다. CSV는 자동으로 읽습니다. 실패하면 수업 페이지의 students-v1.csv를 코랩 왼쪽 파일 영역에 올리세요.

**남길 증거:** 예상 한 문장 → 조건별 실제 결과 → 틀린 사례 하나 → 내 판단.

[수업 페이지](https://learning-experiments-gc261103.incheon-seok-1409.chatgpt.site/class/20261117/algorithms/tree.html) · [데이터 출처](https://archive.ics.uci.edu/dataset/320/student+performance)

## 1. 실행 전에 예상하기

아래 값을 바꾸기 전에 어느 설정이 학습 자료와 검증 자료에서 유리할지 자기 말로 적어 보세요.

학습 자료에서 잘 맞히는 것과 처음 보는 자료에서 잘 맞히는 것은 같은가요?

In [ ]:
from pathlib import Path
import hashlib, io, json, platform, urllib.request
import numpy as np
import pandas as pd
import sklearn
try:
    from IPython.display import display
except ImportError:
    display = print
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix
from sklearn.inspection import permutation_importance

FEATURES = ["studytime", "failures", "absences", "schoolsup", "internet"]
TARGET = "below_10"
print("Python", platform.python_version(), "| scikit-learn", sklearn.__version__)

In [ ]:
# 공통 CSV를 자동으로 읽습니다. 같은 이름의 로컬 파일이 있으면 먼저 사용합니다.
DATA_URL = "https://learning-experiments-gc261103.incheon-seok-1409.chatgpt.site/class/20261117/downloads/students-v1.csv"
EXPECTED_SHA256 = "5e01746d0128a964c31cb02597fc1176c182bd502c31a2f669aa8ab0f16de387"
DATA_VERSION = "gc-20261117-v1"
SPLIT_ID = "gc-20261117-stratified-60-20-20-v1"
csv_path = Path("students-v1.csv")
if csv_path.exists():
    data_bytes = csv_path.read_bytes()
else:
    try:
        with urllib.request.urlopen(DATA_URL, timeout=30) as response:
            data_bytes = response.read()
    except Exception as error:
        raise RuntimeError("자동 다운로드에 실패했습니다. 수업 페이지에서 students-v1.csv를 내려받아 코랩 왼쪽 파일 영역에 올린 뒤 이 셀을 다시 실행하세요.") from error
actual_hash = hashlib.sha256(data_bytes).hexdigest()
if actual_hash != EXPECTED_SHA256:
    raise ValueError("수업용 CSV 버전이 다릅니다. 원본 students-v1.csv를 다시 받아 사용하세요.")
df = pd.read_csv(io.BytesIO(data_bytes), encoding="utf-8-sig")
assert len(df) == 649 and df.row_id.is_unique
assert set(df[TARGET]) == {0, 1} and set(df.split) == {"train", "validation", "test"}
display(df.head())
display(df.groupby("split")[TARGET].agg(자료수="size", 성적10점미만="sum"))
print("studytime: 1=<2시간, 2=2~5시간, 3=5~10시간, 4=>10시간 (주당)")
print("아직 test 자료로 모델을 평가하지 않습니다.")

## 2. 비교 조건 맞추기

파일의 train / validation / test 표시를 그대로 사용합니다. row_id, split, below_10은 입력 항목이 아닙니다. 전처리도 학습 자료에만 fit합니다. studytime과 failures는 범주로 변환하고 absences만 표준화합니다.

In [ ]:
train = df.loc[df.split == "train"].copy()
validation = df.loc[df.split == "validation"].copy()
test = df.loc[df.split == "test"].copy()
assert not (set(train.row_id) & set(validation.row_id) or set(train.row_id) & set(test.row_id) or set(validation.row_id) & set(test.row_id))

def build_model(algorithm, value, columns=FEATURES):
    # studytime과 failures의 구간 코드를 연속 시간/횟수라고 가정하지 않습니다.
    numeric = [c for c in ["absences"] if c in columns]
    categories = [c for c in columns if c not in numeric]
    transforms = []
    if numeric:
        transforms.append(("numeric", StandardScaler(), numeric))
    if categories:
        transforms.append(("category", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categories))
    preprocessor = ColumnTransformer(transforms, remainder="drop")
    estimators = {
        "logistic": lambda: LogisticRegression(C=value, max_iter=3000, random_state=42),
        "tree": lambda: DecisionTreeClassifier(max_depth=value, random_state=42),
        "forest": lambda: RandomForestClassifier(n_estimators=100, max_depth=value, random_state=42, n_jobs=1),
        "knn": lambda: KNeighborsClassifier(n_neighbors=value),
        "svm": lambda: SVC(C=value, kernel="rbf", gamma="scale"),
    }
    return Pipeline([("prepare", preprocessor), ("model", estimators[algorithm]())])

def metrics(y_true, prediction):
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    return dict(n=int(tn+fp+fn+tp), accuracy=float(accuracy_score(y_true, prediction)),
                balanced_accuracy=float(balanced_accuracy_score(y_true, prediction)),
                recall_below_10=float(tp/(tp+fn)) if tp+fn else None,
                recall_at_least_10=float(tn/(tn+fp)) if tn+fp else None,
                missed_below_10=int(fn), false_flags=int(fp),
                tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp))

def evaluate(model, data=validation, columns=FEATURES):
    return metrics(data[TARGET], model.predict(data[columns]))

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(train[FEATURES], train[TARGET])
baseline_metrics = evaluate(baseline)
display(pd.DataFrame([baseline_metrics], index=["다수 범주만 답하는 기준 모델"]))
print("정확도만 보고 성공이라고 말할 수 있을까요? 10점 미만을 놓친 수를 함께 보세요.")

## 3. max_depth 하나만 바꾸기

깊이가 깊을수록 더 복잡한 분류 규칙을 만들 수 있습니다. None은 깊이 제한이 없다는 뜻입니다.

각 실행의 검증 대상은 같습니다. 먼저 학습 점수와 검증 점수가 함께 움직이는지 살펴보세요.

In [ ]:
ALGORITHM = 'tree'
ALGORITHM_NAME = '결정트리'
PARAMETER_NAME = 'max_depth'
PARAMETER_VALUES = [2, 4, None]
DEFAULT_VALUE = 4

In [ ]:
runs = []
models = {}
for value in PARAMETER_VALUES:
    model = build_model(ALGORITHM, value)
    model.fit(train[FEATURES], train[TARGET])
    result = {"parameter": PARAMETER_NAME, "value": value,
              "train_accuracy": float(accuracy_score(train[TARGET], model.predict(train[FEATURES]))),
              **evaluate(model)}
    runs.append(result)
    models[str(value)] = model
display(pd.DataFrame(runs).round(4))
print("기준 모델 검증 정확도:", round(baseline_metrics["accuracy"], 4))
print("값을 고르는 기준은 정확도인가요, 10점 미만 사례를 놓치지 않는 것인가요?")

## 4. 틀린 사례와 항목의 중요도 보기

실제 10점 미만인데 놓친 사례를 찾아보세요. 그 학생의 성격이나 노력 부족을 데이터에 없는 정보로 추측하지 않습니다. 중요도는 이 모델의 의존도이며 인과관계가 아닙니다.

In [ ]:
# 자동으로 1등을 고르지 않습니다. 우선 페이지에 지정된 기본 설정을 살펴봅니다.
SELECTED_VALUE = DEFAULT_VALUE
selected_model = models[str(SELECTED_VALUE)]
pred = selected_model.predict(validation[FEATURES])
cases = validation[["row_id"] + FEATURES + [TARGET]].copy()
cases["prediction"] = pred
cases["wrong"] = cases[TARGET] != cases.prediction
cases["error_type"] = np.where((cases[TARGET] == 1) & (cases.prediction == 0), "10점 미만 놓침",
                              np.where((cases[TARGET] == 0) & (cases.prediction == 1), "10점 이상 잘못 표시", "일치"))
display(cases.loc[cases.wrong].head(10))
print("틀린 사례가 없더라도 다른 자료에서도 완벽하다는 뜻은 아닙니다.")
importance = permutation_importance(selected_model, validation[FEATURES], validation[TARGET],
                                    scoring="balanced_accuracy", n_repeats=10, random_state=42)
display(pd.DataFrame({"항목": FEATURES, "검증점수_감소_평균": importance.importances_mean,
                      "흔들림_표준편차": importance.importances_std}).sort_values("검증점수_감소_평균", ascending=False))
print("이 중요도는 이 모델과 자료에서의 의존도를 뜻합니다. 학생 성적의 원인이나 지원의 효과를 증명하지 않습니다.")

## 5. 지난 실습과 연결하기

학습 자료 20명 / 학습 정답 20% 오류 / 결석 항목 제외를 비교합니다. 변경 조건은 서로 누적하지 않습니다. 결과가 반드시 나빠진다고 가정하지 않습니다.

In [ ]:
# 세 조건은 매번 원본 학습 자료에서 따로 시작합니다. 검증 대상과 정답은 그대로 둡니다.
stress_rows = []
small = train.groupby(TARGET, group_keys=False).sample(n=10, random_state=42)
small_model = build_model(ALGORITHM, DEFAULT_VALUE)
small_model.fit(small[FEATURES], small[TARGET])
stress_rows.append({"condition": "학습 20명(범주별 10명)", **evaluate(small_model)})

noisy = train.copy()
flip_ids = noisy.sample(frac=0.2, random_state=42).index
noisy.loc[flip_ids, TARGET] = 1 - noisy.loc[flip_ids, TARGET]
noisy_model = build_model(ALGORITHM, DEFAULT_VALUE)
noisy_model.fit(noisy[FEATURES], noisy[TARGET])
stress_rows.append({"condition": f"학습 정답 {len(flip_ids)}개 뒤집기", **evaluate(noisy_model)})

reduced_features = [c for c in FEATURES if c != "absences"]
reduced_model = build_model(ALGORITHM, DEFAULT_VALUE, reduced_features)
reduced_model.fit(train[reduced_features], train[TARGET])
stress_rows.append({"condition": "결석 항목 제외", **evaluate(reduced_model, columns=reduced_features)})
display(pd.DataFrame([{"condition": "기본 설정", **evaluate(models[str(DEFAULT_VALUE)])}] + stress_rows).round(4))
print("20명 실험은 학습량과 범주 비율이 함께 바뀝니다. 차이를 자료 수만의 효과라고 단정하지 마세요.")
print("오류나 항목 제거가 이번 분할에서 반드시 점수를 낮추는 것은 아닙니다.")

## 6. 결과 파일 내려받기

result-알고리즘.json을 수업 사이트의 결과 비교 페이지에 올리세요. 업로드한 파일은 현재 브라우저에서만 읽습니다. 검증 대상과 데이터 버전이 다르면 비교에서 제외합니다.

In [ ]:
report = {
    "schema_version": 1, "data_version": DATA_VERSION, "dataset_sha256": actual_hash,
    "split_id": SPLIT_ID, "scope": "validation", "features": FEATURES,
    "algorithm": ALGORITHM, "algorithm_name": ALGORITHM_NAME,
    "baseline": baseline_metrics, "runs": runs, "stress": stress_rows,
    "selected_value": SELECTED_VALUE, "selected_metrics": evaluate(selected_model),
    "validation_row_ids": validation.row_id.tolist(),
    "environment": {"python": platform.python_version(), "sklearn": sklearn.__version__},
}
output = Path(f"result-{ALGORITHM}.json")
output.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
cases.to_csv(f"cases-{ALGORITHM}.csv", index=False, encoding="utf-8-sig")
print("생성 파일:", output.name, f"cases-{ALGORITHM}.csv")
try:
    from google.colab import files
except ImportError:
    print("로컬 실행: 현재 폴더의 파일을 사용하세요.")
else:
    files.download(str(output))

## 7. 내 판단을 먼저 기록하기

- 조건을 바꾸기 전의 예상은 무엇이었나요?
- 실제 결과와 어느 부분이 달랐나요?
- 단순 기준 모델과 비교하면 어떤 오류를 더 줄였나요?
- AI 없이 내 알고리즘을 두 문장으로 설명할 수 있나요?

**작은 확인 문제:** 검증 자료 20명 중 18명이 10점 이상이라면, 모든 학생을 10점 이상으로 답하는 모델의 정확도와 10점 미만 재현율은 각각 얼마인가요?

<details><summary>답 확인</summary>정확도는 90%, 10점 미만 재현율은 0%입니다. 정확도만으로 필요한 사례를 찾았다고 말할 수 없습니다.</details>

## 8. 선택 확정 후에만 최종 확인

검증 자료에서 설정을 고른 후 따로 보관한 test에서 한 번 확인합니다. test 결과를 보고 계속 설정을 고르면 새로운 보관 자료가 필요합니다. 이 소규모 자료에서의 결과를 실제 학습지원 도구의 성능으로 일반화하지 않습니다.

In [ ]:
# 조건을 확정한 뒤에만 True로 바꾸세요. 기본 '모두 실행'에서는 test를 열지 않습니다.
RUN_FINAL_TEST = False
FINAL_VALUE = SELECTED_VALUE  # 검증 결과로 고른 값을 적습니다.
if RUN_FINAL_TEST:
    assert FINAL_VALUE in PARAMETER_VALUES
    final_model = build_model(ALGORITHM, FINAL_VALUE)
    final_model.fit(train[FEATURES], train[TARGET])
    print("선택한 설정:", PARAMETER_NAME, FINAL_VALUE)
    display(pd.DataFrame([evaluate(final_model, data=test)], index=["최종 보관 자료"]))
    print("이 결과를 보고 다시 설정을 바꾸면 test도 다음 선택을 위한 검증 자료가 됩니다.")
else:
    print("test 자료는 아직 평가하지 않았습니다. 먼저 결과에 대한 판단을 기록하세요.")